# PepMLM &rarr; Boltz-2 peptide binder pipeline

Generate linear peptide binders against your own target with
[PepMLM](https://github.com/programmablebio/pepmlm), cofold every candidate with
[Boltz-2](https://github.com/jwohlwend/boltz), and keep only the ones predicted to
bind **where you asked**.

PepMLM on its own gives you sequences and a self-assigned score that correlates with
AlphaFold confidence at only **r = -0.41** — it is an idea generator, not a filter.
This notebook supplies the filter.

**The funnel**

| | | cost |
|---|---|---|
| 1 | Set up everything | ~20 min, once per session |
| 2 | Target: chain and crop | ~2 min (builds the receptor MSA once) |
| 3 | Generate a few thousand peptides and gate on perplexity | minutes |
| 4 | Cofold the survivors, steered at your hotspots | **hours — this is the whole cost** |
| 5 | Filter on interface quality, hotspots and synthesisability | seconds |
| 6 | Look at a result | — |

Steps 3, 4 and 5 are separate on purpose: you can re-filter without re-folding, and
re-fold without re-generating.

**Two files land in your Drive:** `master.csv` is every peptide generated with all of
its cofolding metrics, unfiltered. `final.csv` is the passing set, ranked.

Licences: PepMLM has no licence file in its repo — ask the authors before commercial
use. Boltz-2 is MIT, code and weights, commercial use included.

In [ ]:
#@title 1 &middot; Set up everything { display-mode: "form" }
#@markdown Run once per session. Installs PepMLM and Boltz-2 and fetches every weight
#@markdown the later steps need, so **nothing downloads mid-run**. Safe to re-run:
#@markdown finished phases are skipped and partial downloads resume.
drive_folder = "PepMLM"  #@param {type:"string"}
#@markdown Keep the 4 GB of Boltz weights in Drive, so the next session copies them
#@markdown instead of downloading them again:
cache_weights_in_drive = False  #@param {type:"boolean"}
#@markdown Fetch the 2 GB affinity checkpoint too. It only works on small molecules,
#@markdown never on peptides, so leave this off unless you have another use for it:
fetch_affinity_weights = False  #@param {type:"boolean"}
boltz_version = "2.2.1"  #@param {type:"string"}

import json, os, shutil, subprocess, sys, tarfile, time, urllib.request

T0 = time.time()
WORK   = "/content/pepmlm"
ENV    = "/content/boltz_env"
BCACHE = "/content/boltz_cache"
HFHOME = "/content/hf_cache"
STAMP  = "/content/.pepmlm_env_ok"
PEPMLM_REPO = "ChatterjeeLab/PepMLM-650M"

MOL_URL  = "https://huggingface.co/boltz-community/boltz-2/resolve/main/mols.tar"
CONF_URL = "https://huggingface.co/boltz-community/boltz-2/resolve/main/boltz2_conf.ckpt"
AFF_URL  = "https://huggingface.co/boltz-community/boltz-2/resolve/main/boltz2_aff.ckpt"


def phase(msg):
    print("\n" + "=" * 70)
    print(f"  {msg}    [{(time.time() - T0) / 60:.1f} min in]")
    print("=" * 70, flush=True)


def sh(cmd, env=None, quiet=True, check=True):
    """Run a command, streaming its output unless quiet."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), env=env,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0 or not quiet:
        tail = "\n".join(p.stdout.splitlines()[-40:]) if p.stdout else ""
        print(tail, flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.stdout or ""


# ----------------------------------------------------------------- downloads
def _remote_size(url):
    try:
        req = urllib.request.Request(url, method="HEAD",
                                     headers={"User-Agent": "curl/8"})
        with urllib.request.urlopen(req, timeout=40) as r:
            return int(r.headers.get("Content-Length") or 0)
    except Exception:
        return 0


_barw = [0]


def _progress(label, done, total, elapsed, moved, final=False):
    pct = 100.0 * done / total if total else 0.0
    rate = moved / elapsed if elapsed > 0.5 else 0.0
    eta = (total - done) / rate if rate > 0 and total else 0.0
    fill = int(28 * pct / 100)
    bar = "=" * fill + "-" * (28 - fill)
    txt = (f"  {label[:22]:22s} [{bar}] {pct:5.1f}%  "
           f"{done / 1e6:,.0f}/{total / 1e6:,.0f} MB  {rate / 1e6:4.1f} MB/s")
    txt += f"  done in {elapsed:.0f}s" if final else (
        f"  ~{eta / 60:.0f} min left" if eta > 90 else f"  ~{eta:.0f}s left")
    _barw[0] = max(_barw[0], len(txt))
    sys.stdout.write("\r" + txt.ljust(_barw[0]) + ("\n" if final else ""))
    sys.stdout.flush()
    if final:
        _barw[0] = 0


def fetch(url, path, label):
    """Resumable download that leaves a correctly-sized file alone."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    want = _remote_size(url)
    have = os.path.getsize(path) if os.path.isfile(path) else 0
    if want and have == want:
        print(f"  have   {label[:22]:22s} ({have / 1e6:,.0f} MB)", flush=True)
        return
    if want and have > want:
        os.remove(path)          # longer than the server's copy: start over
        have = 0
    req = urllib.request.Request(url, headers={"User-Agent": "curl/8"})
    if have:
        req.add_header("Range", f"bytes={have}-")
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0             # server ignored the range: restart cleanly
        total = want or (have + int(r.headers.get("Content-Length") or 0))
        t0, last, done = time.time(), 0.0, have
        with open(path, "ab" if have else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if time.time() - last > 0.4:
                    last = time.time()
                    _progress(label, done, total, last - t0, done - have)
    _progress(label, done, total, max(time.time() - t0, 0.1), done - have, final=True)
    if want and os.path.getsize(path) != want:
        raise RuntimeError(f"{label}: got {os.path.getsize(path)} bytes, expected {want}")


def same_size(a, b):
    return (os.path.isfile(a) and os.path.isfile(b)
            and os.path.getsize(a) == os.path.getsize(b))


# ----------------------------------------------------------------- 0. GPU
phase("GPU")
try:
    _q = sh(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
             "--format=csv,noheader"]).strip().splitlines()[0]
    GPU_NAME, _mem, _cc = [x.strip() for x in _q.split(",")]
    GPU_MEM = int(_mem.split()[0])
    GPU_CC = float(_cc)
except Exception as e:
    raise RuntimeError(
        "No GPU found. Runtime > Change runtime type > GPU, then re-run.") from e

HAS_KERNELS = GPU_CC >= 8.0
print(f"  {GPU_NAME}   {GPU_MEM / 1024:.0f} GB   compute {GPU_CC}")
if HAS_KERNELS:
    TIER, N_COFOLD = "fast", 300
    print("  Optimised kernels available. Cofolding will be quick.")
else:
    TIER, N_COFOLD = "slow", 60
    print("  Compute < 8.0, so Boltz runs without its fused kernels (--no_kernels).")
    print(f"  Expect roughly 2-4 min per design on this GPU.")
print(f"  Default number of designs to cofold: {N_COFOLD} (you can change it in step 4)")

# ----------------------------------------------------------------- 1. Drive
phase("Google Drive")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = f"/content/drive/MyDrive/{drive_folder.strip().strip('/')}"
for d in (DRIVE, f"{DRIVE}/inputs", f"{DRIVE}/weights", f"{DRIVE}/receptors"):
    os.makedirs(d, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print(f"  outputs will go to   My Drive/{drive_folder}/<run_name>/")
print(f"  put your PDB in      My Drive/{drive_folder}/inputs/")

# ----------------------------------------------------------------- 2. env
if os.path.exists(STAMP):
    phase("Environment (already built, skipping)")
else:
    phase("Python 3.11 for Boltz-2  (~1 min)")
    # Boltz needs 3.10-3.12; Colab's own interpreter may be outside that range, and
    # an isolated env also keeps Boltz's torch away from the one PepMLM uses.
    sh("apt-get -qq update")
    sh("apt-get -qq install -y python3.11 python3.11-venv python3.11-dev")
    sh(f"python3.11 -m venv {ENV}")
    sh(f"{ENV}/bin/pip install -q --no-cache-dir -U pip wheel")

    phase(f"Boltz-{boltz_version}  (~8 min, ~5 GB)")
    extra = "[cuda]" if HAS_KERNELS else ""
    sh(f"{ENV}/bin/pip install -q --no-cache-dir 'boltz{extra}=={boltz_version}'",
       quiet=False)

    phase("PepMLM dependencies in Colab's own Python  (~1 min)")
    sh(f"{sys.executable} -m pip install -q -U transformers huggingface_hub gemmi py3Dmol")
    open(STAMP, "w").write("ok")

BENV = {**os.environ,
        "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
        "PYTHONPATH": "",                 # Colab exports /env/python; it leaks
        "BOLTZ_CACHE": BCACHE,
        "HF_HOME": HFHOME}
os.environ["HF_HOME"] = HFHOME

# ----------------------------------------------------------------- 3. weights
phase("Boltz-2 weights  (~4 GB)")
os.makedirs(BCACHE, exist_ok=True)
_dw = f"{DRIVE}/weights"
for name, url in (("mols.tar", MOL_URL), ("boltz2_conf.ckpt", CONF_URL)):
    local, cached = f"{BCACHE}/{name}", f"{_dw}/{name}"
    if cache_weights_in_drive and same_size(cached, local):
        print(f"  have   {name[:22]:22s} (local)")
        continue
    if cache_weights_in_drive and os.path.isfile(cached) \
            and os.path.getsize(cached) == _remote_size(url):
        print(f"  copying {name} from Drive ...", flush=True)
        shutil.copyfile(cached, local)
        continue
    fetch(url, local, name)
    if cache_weights_in_drive:
        print(f"  copying {name} to Drive ...", flush=True)
        shutil.copyfile(local, cached)

# The affinity head is small-molecule only, so it is never loaded by this notebook.
# Boltz downloads it on every run if the file is absent, so leave a placeholder.
_aff = f"{BCACHE}/boltz2_aff.ckpt"
if fetch_affinity_weights:
    fetch(AFF_URL, _aff, "boltz2_aff.ckpt")
elif not os.path.exists(_aff):
    open(_aff, "wb").close()
    print("  skipped boltz2_aff.ckpt (small-molecule only); placeholder written")

if not os.path.isdir(f"{BCACHE}/mols") or not os.listdir(f"{BCACHE}/mols"):
    print("  extracting mols.tar (~2 min) ...", flush=True)
    with tarfile.open(f"{BCACHE}/mols.tar", "r") as tar:
        tar.extractall(BCACHE)
print(f"  component library: {len(os.listdir(f'{BCACHE}/mols')):,} files")

# ----------------------------------------------------------------- 4. PepMLM
phase("PepMLM-650M  (~2.6 GB)")
#@markdown
from huggingface_hub import snapshot_download
_pep = snapshot_download(PEPMLM_REPO, cache_dir=f"{HFHOME}/hub")
print(f"  {PEPMLM_REPO} -> {_pep}")

# ----------------------------------------------------------------- 5. check
phase("Checking the install")
_chk = sh(f"{ENV}/bin/python -c "
          f"\"import torch,boltz;print(torch.__version__,torch.cuda.is_available())\"",
          env=BENV).strip()
print(f"  boltz env: torch {_chk}")
assert _chk.endswith("True"), (
    "Boltz's torch cannot see the GPU. Do not re-run this cell in place -- "
    "Runtime > Disconnect and delete runtime, then start again.")
sh(f"{ENV}/bin/boltz predict --help", env=BENV)
print("  boltz CLI ok")

import torch
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(PEPMLM_REPO, cache_dir=f"{HFHOME}/hub")
print(f"  colab env: torch {torch.__version__}  cuda {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Colab's torch cannot see the GPU."

json.dump({"work": WORK, "env": ENV, "cache": BCACHE, "hf": HFHOME,
           "drive": DRIVE, "drive_folder": drive_folder, "repo": PEPMLM_REPO,
           "gpu": GPU_NAME, "gpu_mem": GPU_MEM, "gpu_cc": GPU_CC,
           "kernels": HAS_KERNELS, "tier": TIER, "n_cofold": N_COFOLD},
          open(f"{WORK}/env.json", "w"), indent=2)

phase(f"Ready in {(time.time() - T0) / 60:.1f} min -- go to step 2")

In [ ]:
#@title 2 &middot; Target { display-mode: "form" }
#@markdown Where the structure comes from. For a Drive file, give just the filename as
#@markdown it sits in `My Drive/<drive_folder>/inputs/`.
target_from = "Google Drive file"  #@param ["Google Drive file", "PDB ID (fetch from RCSB)", "path on disk"]
target_path = "receptor.pdb"  #@param {type:"string"}
target_chain = "A"  #@param {type:"string"}
#@markdown Region to keep, **in your file's own residue numbering** — e.g. `20-140`,
#@markdown or several ranges `20-140,190-230`. Leave empty to use the whole chain.
crop = ""  #@param {type:"string"}
#@markdown Name for this run's output folder (empty = derived from the filename):
run_name = ""  #@param {type:"string"}
use_template = True  #@param {type:"boolean"}
show_structure = True  #@param {type:"boolean"}

import glob, hashlib, json, os, shutil, subprocess, sys, time
import gemmi

E = json.load(open("/content/pepmlm/env.json"))
WORK, DRIVE, ENV, BCACHE = E["work"], E["drive"], E["env"], E["cache"]
BENV = {**os.environ, "PATH": f"{ENV}/bin:" + os.environ.get("PATH", ""),
        "PYTHONPATH": "", "BOLTZ_CACHE": BCACHE, "HF_HOME": E["hf"]}
MAX_ESM = 1024          # ESM-2 position limit; receptor + peptide must fit

# ------------------------------------------------------------------ 1. locate
if target_from.startswith("PDB ID"):
    pid = target_path.strip().lower().replace(".cif", "").replace(".pdb", "")
    assert len(pid) == 4, f"'{target_path}' is not a 4-character PDB id"
    src = f"{WORK}/{pid}.cif"
    if not os.path.isfile(src):
        import urllib.request
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pid}.cif", src)
    print(f"fetched {pid.upper()} from RCSB")
elif target_from.startswith("Google Drive"):
    src = f"{DRIVE}/inputs/{target_path.strip().lstrip('/')}"
    assert os.path.isfile(src), (
        f"not found: My Drive/{E['drive_folder']}/inputs/{target_path}\n"
        f"  files there now: {sorted(os.listdir(f'{DRIVE}/inputs')) or '(none)'}")
else:
    src = target_path.strip()
    assert os.path.isfile(src), f"not found: {src}"

stem = os.path.splitext(os.path.basename(src))[0]
RUN = (run_name.strip() or f"{stem}_{target_chain.strip()}")
RUN = "".join(c if (c.isalnum() or c in "-_.") else "_" for c in RUN)
OUT = f"{DRIVE}/{RUN}"
for d in (OUT, f"{OUT}/structures", f"{OUT}/passing"):
    os.makedirs(d, exist_ok=True)
print(f"run name: {RUN}\noutputs:  My Drive/{E['drive_folder']}/{RUN}/")

# ------------------------------------------------------------------ 2. parse
st = gemmi.read_structure(src)
st.setup_entities()
st.remove_alternative_conformations()
st.remove_hydrogens()
st.remove_ligands_and_waters()
assert len(st) > 0, "no models in the structure"
model = st[0]

chains = [c.name for c in model]
ch_name = target_chain.strip()
assert ch_name in chains, f"chain '{ch_name}' not in this file. Chains: {chains}"

res = []                 # (auth_label, one-letter, gemmi residue)
for r in model[ch_name]:
    info = gemmi.find_tabulated_residue(r.name)
    if not (info and info.is_amino_acid()):
        continue
    if not r.find_atom("CA", "*"):
        continue
    letter = (info.one_letter_code or "x").upper()
    res.append((f"{r.seqid.num}{r.seqid.icode.strip()}", letter, r))
assert res, f"chain {ch_name} has no amino-acid residues with a CA atom"
print(f"chain {ch_name}: {len(res)} residues, "
      f"numbered {res[0][0]} to {res[-1][0]}")


def _ranges(text):
    out = []
    for part in text.replace(" ", "").split(","):
        if not part:
            continue
        if "-" in part.lstrip("-"):
            a, b = part.split("-", 1)
            out.append((int(a), int(b)))
        else:
            out.append((int(part), int(part)))
    return out


# ------------------------------------------------------------------ 3. crop
if crop.strip():
    keep = _ranges(crop)
    sel = [t for t in res
           if any(a <= int("".join(ch for ch in t[0] if ch.isdigit() or ch == "-")) <= b
                  for a, b in keep)]
    assert sel, f"crop {crop} selected nothing from chain {ch_name}"
    print(f"crop {crop}: kept {len(sel)} of {len(res)} residues")
    res = sel

SEQ = "".join(t[1] for t in res)
if "X" in SEQ:
    print(f"note: {SEQ.count('X')} non-standard residue(s) became X")

# gaps: residues the file does not observe are silently joined together
gaps = []
for i in range(1, len(res)):
    try:
        a, b = int(res[i - 1][0]), int(res[i][0])
    except ValueError:
        continue
    if b - a > 1:
        gaps.append((res[i - 1][0], res[i][0], b - a - 1))
if gaps:
    print(f"\n!! {len(gaps)} gap(s) in the observed numbering. The sequence sent to "
          f"the models joins them as if continuous:")
    for a, b, n in gaps[:8]:
        print(f"     {a} -> {b}   ({n} residue(s) missing)")
    print("   If a gap sits in the region you care about, crop to a gap-free "
          "stretch instead.")

# ------------------------------------------------------------------ 5. template
tpl = gemmi.Structure()
tpl.name = "receptor"
tpl.spacegroup_hm = "P 1"
m = gemmi.Model("1")
c = gemmi.Chain("A")
for i, (_lab, _let, r) in enumerate(res, start=1):
    nr = gemmi.Residue()
    nr.name, nr.seqid = r.name, gemmi.SeqId(i, " ")
    for a in r:
        nr.add_atom(a)
    c.add_residue(nr)
m.add_chain(c)
tpl.add_model(m)
tpl.setup_entities()
# setup_entities leaves _entity_poly with no sequence, and Boltz aligns the chain
# against that sequence. Against an empty one it runs off the end of the list with
# "IndexError: list index out of range" and silently skips every single design.
for ent in tpl.entities:
    if ent.entity_type == gemmi.EntityType.Polymer:
        ent.full_sequence = [r.name for r in c]
# Boltz names a template's chains by their mmCIF subchain (label_asym_id), not by the
# chain name, and gemmi invents subchain labels like "Axp" -- which Boltz then rejects
# with "Template chain A ... is not one of the protein chains!". Make them match.
for ch in tpl[0]:
    for r in ch:
        r.subchain = ch.name
for ent in tpl.entities:
    ent.subchains = ["A"]
tpl.assign_label_seq_id()
TPL = f"{WORK}/receptor.cif"
tpl.make_mmcif_document().write_file(TPL)

# Read it back exactly as Boltz will, and run the same checks it runs.
_chk = gemmi.make_structure_from_block(gemmi.cif.read(TPL)[0])
_chk.merge_chain_parts()
_chk.remove_waters()
_chk.remove_empty_chains()
_subs = [x.subchain_id() for x in _chk[0].subchains()]
_emap = {sc: e for e in _chk.entities for sc in e.subchains}
assert "A" in _subs, (
    f"the template's chain is labelled {_subs} rather than ['A'], so Boltz would "
    f"refuse it. Report this with your input file.")
assert all(s in _emap for s in _subs), (
    f"the template's chains {_subs} are not all in its entity records {list(_emap)}")
_fs = [gemmi.Entity.first_mon(x) for x in _chk.entities[0].full_sequence]
assert len(_fs) == len(res), (
    f"the template kept only {len(_fs)} of {len(res)} residues in its sequence "
    f"record; Boltz would refuse every design. Report this with your input file.")
_al = gemmi.align_sequence_to_polymer(
    _fs, [x for x in _chk[0].subchains()][0], gemmi.PolymerType.PeptideL,
    gemmi.AlignmentScoring())
assert len(_al.match_string) <= len(_fs), (
    "the template's sequence and its coordinates disagree, so Boltz would refuse it")
print(f"\ntemplate written and verified, chain A renumbered 1..{len(res)}")

# ------------------------------------------------------------------ 6. MSA + apo
H = hashlib.md5(SEQ.encode()).hexdigest()[:12]
CACHE = f"{DRIVE}/receptors/{H}"
MSA, APO = f"{WORK}/receptor_msa.csv", f"{WORK}/receptor_apo.cif"
if os.path.isfile(f"{CACHE}/msa.csv"):
    shutil.copyfile(f"{CACHE}/msa.csv", MSA)
    if os.path.isfile(f"{CACHE}/apo.cif"):
        shutil.copyfile(f"{CACHE}/apo.cif", APO)
    print(f"reusing the cached MSA for this exact sequence ({H})")
else:
    print("\nbuilding the receptor MSA once, to be reused by every design.")
    print("this also predicts the free receptor, which the filter uses later.\n")
    d = f"{WORK}/rec"
    os.makedirs(d, exist_ok=True)
    open(f"{d}/receptor.yaml", "w").write(
        "version: 1\nsequences:\n  - protein:\n      id: A\n"
        f"      sequence: {SEQ}\n")
    cmd = [f"{ENV}/bin/boltz", "predict", f"{d}/receptor.yaml",
           "--out_dir", d, "--use_msa_server", "--cache", BCACHE,
           "--recycling_steps", "3", "--sampling_steps", "50",
           "--diffusion_samples", "1", "--output_format", "mmcif",
           "--num_workers", "2"]
    if not E["kernels"]:
        cmd.append("--no_kernels")
    t0 = time.time()
    p = subprocess.run(cmd, env=BENV, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
    hits = sorted(glob.glob(f"{d}/**/msa/*.csv", recursive=True))
    if not hits:
        print("\n".join(p.stdout.splitlines()[-40:]))
        raise RuntimeError("the MSA server did not return an alignment -- see above")
    shutil.copyfile(hits[0], MSA)
    apo = sorted(glob.glob(f"{d}/**/predictions/**/*_model_0.cif", recursive=True))
    if apo:
        shutil.copyfile(apo[0], APO)
    os.makedirs(CACHE, exist_ok=True)
    shutil.copyfile(MSA, f"{CACHE}/msa.csv")
    if apo:
        shutil.copyfile(APO, f"{CACHE}/apo.cif")
    open(f"{CACHE}/sequence.txt", "w").write(SEQ + "\n")
    print(f"  MSA + free receptor done in {(time.time() - t0) / 60:.1f} min, "
          f"cached as receptors/{H}")

_depth = sum(1 for _ in open(MSA)) - 1
print(f"MSA depth: {_depth:,} sequences")
if _depth < 100:
    print("!! a shallow MSA usually means a sequence the databases barely cover; "
          "Boltz will be less confident on this target")

# ------------------------------------------------------------------ 7. save
json.dump({"run": RUN, "out": OUT, "src": src, "chain": ch_name,
           "seq": SEQ, "n_res": len(res), "crop": crop.strip(),
           "labels": [t[0] for t in res],
           "template": TPL if use_template else "", "msa": MSA, "apo": APO,
           "hash": H, "max_peptide": MAX_ESM - len(SEQ)},
          open(f"{WORK}/target.json", "w"), indent=2)

print(f"\nreceptor is {len(SEQ)} residues, so the longest peptide PepMLM can take "
      f"is {MAX_ESM - len(SEQ)}")
assert len(SEQ) < MAX_ESM - 10, (
    f"the receptor is {len(SEQ)} residues and ESM-2 stops at {MAX_ESM}. "
    f"Crop it to the domain you care about.")

# ------------------------------------------------------------------ 8. view
if show_structure:
    import py3Dmol
    v = py3Dmol.view(width=760, height=460)
    v.addModel(open(TPL).read(), "cif")
    v.setStyle({"cartoon": {"color": "grey", "opacity": 0.85}})
    v.zoomTo()
    v.show()
    print("This is the target as steps 3 and 4 will see it. Hotspots are set in"
          " step 4 (to steer the folding) and step 5 (to judge the result).")

In [ ]:
#@title 3 &middot; Binder settings, and generate with PepMLM { display-mode: "form" }
#@markdown PepMLM makes one peptide of a fixed length at a time, so this sweeps the
#@markdown range. Generation is cheap; cofolding in step 4 is what costs time.
length_min = 10  #@param {type:"slider", min:5, max:40, step:1}
length_max = 25  #@param {type:"slider", min:5, max:40, step:1}
n_per_length = 100  #@param {type:"integer"}
#@markdown `top_k` is how many amino acids each position may be sampled from. 1 is
#@markdown deterministic, 3 is the published default, higher is more diverse:
top_k = 3  #@param {type:"slider", min:1, max:10, step:1}
#@markdown How to pick which peptides go on to cofolding. Perplexity is **not
#@markdown comparable between lengths**, so the top-N gate ranks within each length:
gate = "top N per length"  #@param ["top N per length", "perplexity cutoff", "everything"]
keep_top_n = 6  #@param {type:"integer"}
ppl_cutoff = 4.0  #@param {type:"number"}
precision = "float16"  #@param ["float16", "float32"]
seed = 0  #@param {type:"integer"}

import json, os, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

E = json.load(open("/content/pepmlm/env.json"))
T = json.load(open("/content/pepmlm/target.json"))
WORK, OUT, SEQ = E["work"], T["out"], T["seq"]

assert length_min <= length_max, "length_min is above length_max"
assert length_max <= T["max_peptide"], (
    f"this receptor is {len(SEQ)} residues, so the longest peptide ESM-2 can "
    f"handle is {T['max_peptide']}")

# ------------------------------------------------------------------ model
DT = torch.float16 if precision == "float16" else torch.float32
# Reload rather than upcast: a model already cast down to fp16 cannot be made
# fp32 again without having lost precision.
if "_PEP" in globals() and _PEP[1].dtype != DT and DT == torch.float32:
    del _PEP
if "_PEP" not in globals():
    print("loading PepMLM-650M ...", flush=True)
    _tok = AutoTokenizer.from_pretrained(E["repo"], cache_dir=f"{E['hf']}/hub")
    _mdl = AutoModelForMaskedLM.from_pretrained(E["repo"], cache_dir=f"{E['hf']}/hub")
    _PEP = (_tok, _mdl.to("cuda").eval())
tok, mdl = _PEP
mdl = mdl.to(DT)

# PepMLM samples from the whole vocabulary, which also holds <mask>, <pad>, X and the
# rare letters B/U/Z/O. Those are then dropped when the peptide is decoded, which can
# silently return a peptide shorter than asked for. Restricting the choice to the 20
# standard amino acids keeps every peptide exactly `length` long and synthesisable.
AA = "ACDEFGHIKLMNPQRSTVWY"
_ids = [tok.convert_tokens_to_ids(a) for a in AA]
assert all(i is not None and i >= 0 and i != tok.unk_token_id for i in _ids), \
    f"the tokenizer does not know every amino acid: {dict(zip(AA, _ids))}"
AA_IDS = torch.tensor(_ids, device="cuda")
K = min(int(top_k), len(AA))

TOKBUDGET = 20000 if E["gpu_mem"] > 20000 else 7000


@torch.no_grad()
def sample_length(L, n):
    """One forward pass, then n draws from it.

    PepMLM's own loop re-runs the model for every binder, but the input is the same
    fully-masked sequence each time and the model is deterministic in eval mode, so
    the logits are identical. Sampling n times from one pass is the same computation.
    """
    ids = tok(SEQ + "<mask>" * L, return_tensors="pt").to("cuda")
    logits = mdl(**ids).logits[0]
    pos = (ids["input_ids"][0] == tok.mask_token_id).nonzero(as_tuple=True)[0]
    lg = logits[pos].float()[:, AA_IDS]                 # (L, 20)
    topv, topi = lg.topk(K, dim=-1)
    probs = torch.softmax(topv, dim=-1)
    draw = torch.multinomial(probs, n, replacement=True)  # (L, n)
    chosen = topi.gather(-1, draw)                        # (L, n)
    out = []
    for j in range(n):
        out.append("".join(AA[i] for i in chosen[:, j].tolist()))
    return out


@torch.no_grad()
def perplexities(peptides):
    """Pseudo-perplexity: mask each peptide position in turn, batched across peptides."""
    rows, owner, truth = [], [], []
    for pi, pep in enumerate(peptides):
        ids = tok(SEQ + pep, return_tensors="pt")["input_ids"][0]
        n = len(pep)
        for k in range(n):
            j = len(ids) - 1 - n + k          # last token is <eos>
            r = ids.clone()
            truth.append(int(r[j]))
            r[j] = tok.mask_token_id
            rows.append((r, j))
            owner.append(pi)
    width = len(rows[0][0])
    per_batch = max(1, TOKBUDGET // width)
    loss_sum = np.zeros(len(peptides))
    loss_cnt = np.zeros(len(peptides))
    for s in range(0, len(rows), per_batch):
        sl = rows[s:s + per_batch]
        batch = torch.stack([r for r, _ in sl]).to("cuda")
        at = torch.tensor([j for _, j in sl], device="cuda")
        lg = mdl(input_ids=batch).logits.float()
        lp = torch.log_softmax(lg[torch.arange(len(sl), device="cuda"), at], dim=-1)
        tr = torch.tensor(truth[s:s + per_batch], device="cuda")
        nll = -lp.gather(-1, tr[:, None])[:, 0].cpu().numpy()
        for o, v in zip(owner[s:s + per_batch], nll):
            loss_sum[o] += v
            loss_cnt[o] += 1
    return np.exp(loss_sum / np.maximum(loss_cnt, 1))


# ------------------------------------------------------------------ generate
torch.manual_seed(int(seed))
lengths = list(range(int(length_min), int(length_max) + 1))
print(f"\n{len(lengths)} lengths x {n_per_length} samples "
      f"= {len(lengths) * n_per_length:,} peptides against {len(SEQ)} residues\n")

rows, t0 = [], time.time()
for L in lengths:
    peps = sorted(set(sample_length(L, int(n_per_length))))
    ppl = perplexities(peps)
    order = np.argsort(ppl)
    for rank, i in enumerate(order, start=1):
        rows.append({"peptide_id": f"L{L:02d}_{rank:04d}", "length": L,
                     "sequence": peps[i], "perplexity": round(float(ppl[i]), 4),
                     "rank_in_length": rank})
    el = time.time() - t0
    frac = (lengths.index(L) + 1) / len(lengths)
    print(f"  length {L:2d}: {len(peps):4d} unique, best perplexity "
          f"{ppl[order[0]]:.3f}   [{el / 60:.1f} min, "
          f"~{el / frac * (1 - frac) / 60:.0f} min left]", flush=True)

df = pd.DataFrame(rows)

# ------------------------------------------------------------------ gate
if gate == "everything":
    df["selected"] = True
elif gate == "perplexity cutoff":
    df["selected"] = df["perplexity"] <= float(ppl_cutoff)
else:
    df["selected"] = df["rank_in_length"] <= int(keep_top_n)

n_sel = int(df["selected"].sum())
print(f"\ngenerated {len(df):,} peptides, {n_sel} selected for cofolding")
if n_sel == 0:
    print("!! nothing selected. Raise keep_top_n, or relax the perplexity cutoff -- "
          f"the best perplexity seen was {df['perplexity'].min():.3f}")
mins = n_sel * (3.0 if not E["kernels"] else 0.5)
print(f"   step 4 will take roughly {mins / 60:.1f} h" if mins > 90 else
      f"   step 4 will take roughly {mins:.0f} min")
if n_sel > E["n_cofold"]:
    print(f"!! that is more than the {E['n_cofold']} this GPU is comfortable with. "
          f"Step 4 has a max_designs cap you can use.")

print("\nbest 10 by perplexity, across all lengths "
      "(remember: not comparable between lengths)")
print(df.nsmallest(10, "perplexity")[
    ["peptide_id", "length", "sequence", "perplexity"]].to_string(index=False))

df.to_csv(f"{WORK}/master.csv", index=False)
df.to_csv(f"{OUT}/master.csv", index=False)
json.dump({"length_min": int(length_min), "length_max": int(length_max),
           "n_per_length": int(n_per_length), "top_k": int(top_k), "gate": gate,
           "keep_top_n": int(keep_top_n), "ppl_cutoff": float(ppl_cutoff),
           "precision": precision, "seed": int(seed),
           "receptor_len": len(SEQ),
           "crop": T["crop"], "chain": T["chain"], "source": T["src"]},
          open(f"{OUT}/settings.json", "w"), indent=2)
print(f"\nmaster.csv written to My Drive/{E['drive_folder']}/{T['run']}/")

In [ ]:
#@title 4 &middot; Cofold with Boltz-2 { display-mode: "form" }
#@markdown This is the slow step. It is resumable: re-running skips designs that are
#@markdown already done, so a Colab disconnect costs you one design.
#@markdown
#@markdown `sampling_steps` 50 is a screening setting. Boltz's published default is
#@markdown 200 — worth re-running on the finalists once you have some.
sampling_steps = 50  #@param {type:"integer"}
recycling_steps = 3  #@param {type:"integer"}
diffusion_samples = 1  #@param {type:"integer"}
#@markdown Cap on how many designs to attempt (0 = every one step 3 selected):
max_designs = 0  #@param {type:"integer"}
#@markdown **Hotspot steering.** The residues you want the peptide to bind, in your
#@markdown file's own numbering, e.g. `40,99,107`. This only aims the folding; what the
#@markdown peptide actually touched is judged in step 5. Leave empty to let Boltz put
#@markdown the peptide wherever it likes.
hotspots = ""  #@param {type:"string"}
#@markdown `force` turns the soft hint into a hard potential — leave it off, since a
#@markdown forced pose can look good without being real:
pocket_max_distance = 6.0  #@param {type:"number"}
pocket_force = False  #@param {type:"boolean"}
#@markdown Geometry measured afterwards. `patch_radius` defines the neighbourhood
#@markdown around your hotspots that counts as "the right face":
contact_cutoff = 4.5  #@param {type:"number"}
patch_radius = 10.0  #@param {type:"number"}
chunk_size = 25  #@param {type:"integer"}
#@markdown How many alignment sequences each design is folded against. Boltz's own
#@markdown help text says it subsamples to 1024 by default, but the flag that does it
#@markdown defaults to off, so without this it uses up to 8192 — several times the
#@markdown work for very little gain. Untick only if you want upstream's raw behaviour.
subsample_msa = True  #@param {type:"boolean"}
msa_depth = 1024  #@param {type:"integer"}
redo = False  #@param {type:"boolean"}

import glob, json, os, re, shutil, subprocess, sys, time
import gemmi
import numpy as np
import pandas as pd

E = json.load(open("/content/pepmlm/env.json"))
T = json.load(open("/content/pepmlm/target.json"))
WORK, OUT, SEQ = E["work"], T["out"], T["seq"]
BENV = {**os.environ, "PATH": f"{E['env']}/bin:" + os.environ.get("PATH", ""),
        "PYTHONPATH": "", "BOLTZ_CACHE": E["cache"], "HF_HOME": E["hf"]}

# Hotspots, in your own numbering, translated to positions in the sequence that
# step 2 built. They steer the folding only.
_given = [x for x in hotspots.replace(" ", "").split(",") if x]
_ix = {lab: i + 1 for i, lab in enumerate(T["labels"])}
_miss = [p for p in _given if p not in _ix]
assert not _miss, (
    f"residue(s) {_miss} are not in the target step 2 built. Its numbering runs "
    f"{T['labels'][0]} .. {T['labels'][-1]}, and a crop in step 2 may have removed them.")
HOT = [_ix[p] for p in _given]
if HOT:
    print("steering towards:")
    for p, i in zip(_given, HOT):
        print(f"   {p:>6s}  ->  {SEQ[i - 1]}{i}")
else:
    print("no hotspots: Boltz will place the peptide wherever it prefers")

df = pd.read_csv(f"{WORK}/master.csv")
todo = df[df["selected"]].copy()
assert len(todo), "step 3 selected nothing to cofold"
_want = len(todo)

# Skip anything already folded in an earlier session: its metrics are in master.csv
# and its structure is in Drive, so there is nothing to recompute. This is what makes
# a fresh session cheap after a disconnect, since the local predictions are gone.
if not redo and "cofolded" in df.columns:
    _have = {p for p in df.loc[df["cofolded"].eq(True), "peptide_id"]
             if os.path.isfile(f"{OUT}/structures/{p}.cif")}
    if _have:
        todo = todo[~todo["peptide_id"].isin(_have)]
        print(f"{len(_have)} design(s) already folded and saved in Drive, skipping them"
              f"{'  (tick redo to fold them again)' if len(_have) else ''}")
if max_designs and len(todo) > int(max_designs):
    todo = todo.nsmallest(int(max_designs), "perplexity")

YD, CD = f"{WORK}/yaml", f"{WORK}/cofold"
if redo:
    shutil.rmtree(YD, ignore_errors=True)
    shutil.rmtree(CD, ignore_errors=True)
os.makedirs(YD, exist_ok=True)
os.makedirs(CD, exist_ok=True)

# --------------------------------------------------------- the hotspot patch
# Receptor residues whose CA sits within patch_radius of any hotspot CA. A peptide
# that contacts the receptor only outside this patch is binding the wrong face.
_tpl = gemmi.read_structure(T["template"])
_ca = {}
for r in _tpl[0]["A"]:
    a = r.find_atom("CA", "*")
    if a:
        _ca[r.seqid.num] = np.array([a.pos.x, a.pos.y, a.pos.z])
if HOT:
    _h = np.array([_ca[i] for i in HOT if i in _ca])
    PATCH = {i for i, p in _ca.items()
             if np.min(np.linalg.norm(_h - p, axis=1)) <= float(patch_radius)}
else:
    PATCH = set()
print(f"{len(SEQ)} residues, {len(HOT)} hotspot(s), "
      f"{len(PATCH)} residues in the hotspot patch ({patch_radius} A)")

# --------------------------------------------------------- write the inputs
def write_yaml(path, pep):
    y = ["version: 1", "sequences:",
         "  - protein:", "      id: A", f"      sequence: {SEQ}",
         f"      msa: {T['msa']}",
         "  - protein:", "      id: B", f"      sequence: {pep}",
         "      msa: empty"]
    if T["template"]:
        y += ["templates:", f"  - cif: {T['template']}",
              "    chain_id: A", "    template_id: A"]
    if HOT:
        y += ["constraints:", "  - pocket:", "      binder: B",
              "      contacts: [" + ", ".join(f"[A, {i}]" for i in HOT) + "]",
              f"      max_distance: {float(pocket_max_distance)}",
              f"      force: {'true' if pocket_force else 'false'}"]
    open(path, "w").write("\n".join(y) + "\n")


ids = list(todo["peptide_id"])
chunks = [ids[i:i + int(chunk_size)] for i in range(0, len(ids), int(chunk_size))]
seqof = dict(zip(todo["peptide_id"], todo["sequence"]))
for ci, ch in enumerate(chunks):
    d = f"{YD}/c{ci:03d}"
    os.makedirs(d, exist_ok=True)
    for pid in ch:
        write_yaml(f"{d}/{pid}.yaml", seqof[pid])

# --------------------------------------------------------- measure one design
def interface(cif, L):
    """Contacts, the hotspot patch overlap, and how far the receptor moved."""
    st = gemmi.read_structure(cif)
    st.remove_hydrogens()
    m = st[0]
    rec = {c.name: c for c in m}
    A, B = rec.get("A"), rec.get("B")
    if A is None or B is None:
        return {}
    ra, rid = [], []
    for r in A:
        for a in r:
            if a.element != gemmi.Element("H"):
                ra.append([a.pos.x, a.pos.y, a.pos.z])
                rid.append(r.seqid.num)
    pb = [[a.pos.x, a.pos.y, a.pos.z] for r in B for a in r
          if a.element != gemmi.Element("H")]
    if not ra or not pb:
        return {}
    ra, pb, rid = np.array(ra), np.array(pb), np.array(rid)
    d = np.linalg.norm(ra[:, None, :] - pb[None, :, :], axis=-1)
    hit = rid[(d <= float(contact_cutoff)).any(axis=1)]
    touched = sorted(set(int(x) for x in hit))
    out = {"n_contact_res": len(touched),
           "contact_residues": ";".join(str(x) for x in touched[:40])}
    out["hotspot_contacts"] = len([i for i in HOT if i in touched])
    out["hotspot_fraction"] = (round(len([i for i in touched if i in PATCH])
                                     / len(touched), 3) if touched and PATCH else 0.0)
    # how much the receptor had to move relative to the free prediction
    out["receptor_rmsd"] = np.nan
    if os.path.isfile(T["apo"]):
        try:
            ap = gemmi.read_structure(T["apo"])
            p1, p2 = [], []
            apo_ca = {r.seqid.num: r.find_atom("CA", "*") for r in ap[0]["A"]}
            for r in A:
                a = r.find_atom("CA", "*")
                b = apo_ca.get(r.seqid.num)
                if a and b:
                    p1.append(a.pos)
                    p2.append(b.pos)
            if len(p1) > 10:
                out["receptor_rmsd"] = round(
                    gemmi.superpose_positions(p1, p2).rmsd, 3)
        except Exception:
            pass
    return out


def collect(pid, pred_dir, L):
    g = f"{pred_dir}/{pid}"
    cif = f"{g}/{pid}_model_0.cif"
    cj = f"{g}/confidence_{pid}_model_0.json"
    if not (os.path.isfile(cif) and os.path.isfile(cj)):
        return None
    c = json.load(open(cj))
    pair = c.get("pair_chains_iptm", {})
    try:
        ip = float(pair["0"]["1"])
    except Exception:
        ip = float(c.get("iptm", np.nan))
    row = {"cofolded": True,
           "iptm": round(float(c.get("iptm", np.nan)), 4),
           "iptm_pair": round(ip, 4),
           "ptm": round(float(c.get("ptm", np.nan)), 4),
           "confidence_score": round(float(c.get("confidence_score", np.nan)), 4),
           "complex_plddt": round(float(c.get("complex_plddt", np.nan)), 4),
           "complex_iplddt": round(float(c.get("complex_iplddt", np.nan)), 4),
           "complex_ipde": round(float(c.get("complex_ipde", np.nan)), 4)}
    pl = f"{g}/plddt_{pid}_model_0.npz"
    if os.path.isfile(pl):
        v = np.load(pl)["plddt"].reshape(-1).astype(float)
        if v.size and v.max() > 1.5:
            v = v / 100.0                      # some builds report pLDDT on 0-100
        if len(v) >= len(SEQ) + L:
            row["pep_plddt"] = round(float(np.mean(v[len(SEQ):len(SEQ) + L])), 4)
    pa = f"{g}/pae_{pid}_model_0.npz"
    if os.path.isfile(pa):
        p = np.load(pa)["pae"]
        p = p[0] if p.ndim == 3 else p
        if p.shape[0] >= len(SEQ) + L:
            a, b = slice(0, len(SEQ)), slice(len(SEQ), len(SEQ) + L)
            row["ipae"] = round(float((p[a, b].mean() + p[b, a].mean()) / 2), 4)
    row.update(interface(cif, L))
    row["cif_name"] = f"{pid}.cif"
    shutil.copyfile(cif, f"{OUT}/structures/{pid}.cif")
    return row


# --------------------------------------------------------- run
_PROG = re.compile(r"(Predicting|Generating|Processing).*?(\d+)%")
_NUM = ["iptm", "iptm_pair", "ptm", "confidence_score", "complex_plddt",
        "complex_iplddt", "complex_ipde", "pep_plddt", "ipae", "n_contact_res",
        "hotspot_contacts", "hotspot_fraction", "receptor_rmsd"]


def tidy(d):
    """Columns created empty and filled in later stay a generic object dtype, which
    pandas will not sort or rank on. Give them their real types back."""
    for c in _NUM:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    if "cofolded" in d.columns:
        d["cofolded"] = d["cofolded"].eq(True)
    return d
done_before = int(df.get("cofolded", pd.Series(dtype=bool)).fillna(False).sum()) \
    if "cofolded" in df else 0
_depth = max(sum(1 for _ in open(T["msa"])) - 1, 0)
print(f"\n{len(ids)} design(s) in {len(chunks)} chunk(s) of {chunk_size}; "
      f"{done_before} already done in master.csv")
print(f"alignment depth {_depth:,}"
      + (f", subsampled to {int(msa_depth):,}" if subsample_msa and _depth > msa_depth
         else "")
      + f"   |   {int(sampling_steps)} sampling steps, "
        f"{int(recycling_steps)} recycles, {int(diffusion_samples)} sample(s)")
print("the progress bar only moves when a design finishes, so it sits at 0% through "
      "the first one\n")

t0, finished = time.time(), 0
for ci, ch in enumerate(chunks):
    d, od = f"{YD}/c{ci:03d}", f"{CD}/c{ci:03d}"
    pred = f"{od}/boltz_results_c{ci:03d}/predictions"
    # Boltz treats a design as done if its output FOLDER exists, without looking
    # inside. A run cut off mid-write leaves an empty folder, which it would then skip
    # for ever. Clear those out so an interrupted design is actually retried.
    _part = [p for p in ch if os.path.isdir(f"{pred}/{p}")
             and not os.path.isfile(f"{pred}/{p}/confidence_{p}_model_0.json")]
    for p in _part:
        shutil.rmtree(f"{pred}/{p}", ignore_errors=True)
    if _part:
        print(f"chunk {ci + 1}: cleared {len(_part)} half-written design(s) so they "
              f"are retried")
    have = sum(1 for p in ch
               if os.path.isfile(f"{pred}/{p}/confidence_{p}_model_0.json"))
    if have == len(ch):
        print(f"chunk {ci + 1}/{len(chunks)}: already complete")
    else:
        cmd = [f"{E['env']}/bin/boltz", "predict", d, "--out_dir", od,
               "--cache", E["cache"], "--recycling_steps", str(int(recycling_steps)),
               "--sampling_steps", str(int(sampling_steps)),
               "--diffusion_samples", str(int(diffusion_samples)),
               "--output_format", "mmcif", "--write_full_pae",
               "--num_workers", "2", "--preprocessing-threads", "2"]
        if subsample_msa:
            cmd += ["--subsample_msa", "--num_subsampled_msa", str(int(msa_depth))]
        if pocket_force:
            cmd.append("--use_potentials")
        if not E["kernels"]:
            cmd.append("--no_kernels")
        print(f"chunk {ci + 1}/{len(chunks)}: {len(ch) - have} to run ...", flush=True)
        p = subprocess.Popen(cmd, env=BENV, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        tail, w, refused, shown = [], 0, [], 0
        for line in p.stdout:
            tail.append(line.rstrip())
            tail[:] = tail[-60:]
            s = line.strip()
            # tqdm writes its bar with a bare \r, so a failure message often arrives
            # appended to the end of a progress line. Check for it first.
            if "Failed to process" in s:
                refused.append(s.split("Error:")[-1].strip().rstrip(".") or s)
            elif _PROG.search(s):
                txt = f"    {s[:110]}"
                w = max(w, len(txt))
                sys.stdout.write("\r" + txt.ljust(w))
                sys.stdout.flush()
            elif "Traceback" in s or s.split(":")[0].endswith("Error"):
                pass                      # already counted above, and very repetitive
            elif "rror" in s and shown < 5:
                shown += 1
                sys.stdout.write("\n    " + s + "\n")
        p.wait()
        sys.stdout.write("\n")
        if refused:
            from collections import Counter
            print(f"    Boltz refused {len(refused)} of {len(ch)} input file(s):")
            for why, n in Counter(refused).most_common(3):
                print(f"      {n} x  {why}")
        if p.returncode != 0:
            print("\n".join(tail))
            raise RuntimeError(
                f"boltz failed on chunk {ci} (exit {p.returncode}). If this says out "
                f"of memory, lower chunk_size or crop the receptor further.")

    got = {}
    for pid in ch:
        r = collect(pid, pred, int(df.loc[df.peptide_id == pid, "length"].iloc[0]))
        if r:
            got[pid] = r
    if not got:
        # Stop here rather than march through every chunk producing nothing.
        print("\n".join(tail[-25:]) if "tail" in dir() else "")
        raise RuntimeError(
            f"none of the {len(ch)} designs in chunk {ci} produced output.\n"
            f"  Boltz's own last lines are above. The inputs it read are in {d},\n"
            f"  and a single one can be retried by hand with:\n"
            f"    !{E['env']}/bin/boltz predict {d}/{ch[0]}.yaml --out_dir /content/try "
            f"--cache {E['cache']}")
    for pid, r in got.items():
        for k, v in r.items():
            if k not in df.columns:
                df[k] = pd.NA
            df.loc[df.peptide_id == pid, k] = v
    df = tidy(df)
    df.to_csv(f"{WORK}/master.csv", index=False)
    df.to_csv(f"{OUT}/master.csv", index=False)
    finished += len(got)
    el = time.time() - t0
    rate = el / max(finished, 1)
    left = (len(ids) - (ci + 1) * int(chunk_size))
    print(f"    {len(got)}/{len(ch)} parsed   {rate:.0f} s/design   "
          f"{el / 60:.0f} min elapsed"
          + (f", ~{max(left, 0) * rate / 60:.0f} min left" if left > 0 else ""))

df = tidy(df)
ok = df["cofolded"] if "cofolded" in df else pd.Series(False, index=df.index)
print(f"\n{int(ok.sum())} of {_want} selected design(s) now have results"
      + (f"; {len(ids)} folded here in {(time.time() - t0) / 60:.1f} min"
         if ids else "; nothing needed folding"))
if int(ok.sum()) < _want:
    print(f"   {_want - int(ok.sum())} still missing. Re-run this cell: finished "
          f"designs are skipped and half-written ones are cleared and retried.")
cols = [c for c in ["peptide_id", "length", "sequence", "perplexity", "iptm_pair",
                    "ipae", "pep_plddt", "hotspot_contacts", "hotspot_fraction"]
        if c in df.columns]
if int(ok.sum()) and "iptm_pair" in df.columns:
    print("\nbest 10 by interface ipTM:")
    print(df[ok].nlargest(10, "iptm_pair")[cols].to_string(index=False))
print(f"\nstructures -> My Drive/{E['drive_folder']}/{T['run']}/structures/")

In [ ]:
#@title 5 &middot; Filter { display-mode: "form" }
#@markdown Re-run this as often as you like. It refolds nothing: every number it
#@markdown needs is already in `master.csv` or in the saved structures, so even
#@markdown changing the hotspots here is free. Each design keeps a `failed`
#@markdown column naming the gates it missed.
#@markdown
#@markdown **Structure gates.** ipTM is the interface confidence for the
#@markdown receptor-peptide pair; ipAE is the interface error in angstroms and is the
#@markdown sharpest single number here.
min_iptm = 0.6  #@param {type:"number"}
max_ipae = 10.0  #@param {type:"number"}
min_pep_plddt = 0.7  #@param {type:"number"}
max_receptor_rmsd = 0.0  #@param {type:"number"}
#@markdown **Hotspot gates.** `min_hotspot_contacts` is how many of your residues the
#@markdown peptide must touch. `min_hotspot_fraction` is the share of its contacts
#@markdown landing on the patch around them — this is what removes a peptide that
#@markdown binds confidently to the wrong face.
min_hotspot_contacts = 2  #@param {type:"integer"}
min_hotspot_fraction = 0.6  #@param {type:"number"}
#@markdown The residues the peptide must actually have bound, in **your file's own
#@markdown numbering**, e.g. `40,99,107`. Independent of step 4's steering: change it
#@markdown and the contacts are re-measured from the structures already saved, so
#@markdown nothing is refolded. Leave empty to skip both hotspot gates.
hotspots = ""  #@param {type:"string"}
patch_radius = 10.0  #@param {type:"number"}
contact_cutoff = 4.5  #@param {type:"number"}
#@markdown **Sequence gates**, for whether you can actually make the peptide.
max_cys = 1  #@param {type:"integer"}
charge_min = -3.0  #@param {type:"number"}
charge_max = 5.0  #@param {type:"number"}
max_gravy = 1.0  #@param {type:"number"}
drop_liabilities = False  #@param {type:"boolean"}
rank_by = "iptm_pair"  #@param ["iptm_pair", "ipae", "confidence_score", "perplexity", "pep_plddt"]

import json, os, re, shutil
import gemmi
import numpy as np
import pandas as pd

E = json.load(open("/content/pepmlm/env.json"))
T = json.load(open("/content/pepmlm/target.json"))
WORK, OUT = E["work"], T["out"]
df = pd.read_csv(f"{WORK}/master.csv")
assert "cofolded" in df.columns, "run step 4 first"
d = df[df["cofolded"].eq(True)].copy()
assert len(d), "nothing has been cofolded yet"

for c in ("iptm", "iptm_pair", "ptm", "confidence_score", "complex_plddt",
          "complex_iplddt", "complex_ipde", "pep_plddt", "ipae", "n_contact_res",
          "hotspot_contacts", "hotspot_fraction", "receptor_rmsd"):
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# plddt is reported on 0-1 by Boltz; tolerate a 0-100 file
for c in ("pep_plddt", "complex_plddt", "complex_iplddt"):
    if c in d.columns and d[c].max() > 1.5:
        d[c] = d[c] / 100.0

# ------------------------------------------------------------ hotspots
LAB = T["labels"]
_given = [x for x in hotspots.replace(" ", "").split(",") if x]
if _given:
    _ix = {lab: i + 1 for i, lab in enumerate(LAB)}
    _miss = [p for p in _given if p not in _ix]
    assert not _miss, (
        f"residue(s) {_miss} are not in the target step 2 built. Its numbering runs "
        f"{LAB[0]} .. {LAB[-1]}, and a crop in step 2 may have removed them.")
    HOT = [_ix[p] for p in _given]
    print("hotspots for this filter, overriding step 2:")
    for p, i in zip(_given, HOT):
        print(f"   {p:>6s}  ->  {T['seq'][i - 1]}{i}")
else:
    HOT = []
    print("no hotspots set here, so the two hotspot gates are skipped")

# ------------------------------------------------------------ re-measure geometry
# Done here rather than reused from step 4, so the hotspots, the patch radius and the
# contact cutoff can all be changed without refolding a single design.
PATCH = set()
if HOT and T["template"] and os.path.isfile(T["template"]):
    _t = gemmi.read_structure(T["template"])
    _ca = {}
    for r in _t[0]["A"]:
        a = r.find_atom("CA", "*")
        if a:
            _ca[r.seqid.num] = np.array([a.pos.x, a.pos.y, a.pos.z])
    _h = np.array([_ca[i] for i in HOT if i in _ca])
    if len(_h):
        PATCH = {i for i, p in _ca.items()
                 if np.min(np.linalg.norm(_h - p, axis=1)) <= float(patch_radius)}


def measure(cif):
    st = gemmi.read_structure(cif)
    st.remove_hydrogens()
    ch = {c.name: c for c in st[0]}
    if "A" not in ch or "B" not in ch:
        return None
    ra, rid = [], []
    for r in ch["A"]:
        for a in r:
            ra.append([a.pos.x, a.pos.y, a.pos.z])
            rid.append(r.seqid.num)
    pb = [[a.pos.x, a.pos.y, a.pos.z] for r in ch["B"] for a in r]
    if not ra or not pb:
        return None
    ra, pb, rid = np.array(ra), np.array(pb), np.array(rid)
    dist = np.linalg.norm(ra[:, None, :] - pb[None, :, :], axis=-1)
    touched = sorted({int(x) for x in rid[(dist <= float(contact_cutoff)).any(axis=1)]})
    return {"n_contact_res": len(touched),
            "contact_residues": ";".join(str(x) for x in touched),
            "hotspot_contacts": len([i for i in HOT if i in touched]),
            "hotspot_fraction": (round(len([i for i in touched if i in PATCH])
                                       / len(touched), 3)
                                 if touched and PATCH else 0.0)}


for c in ("n_contact_res", "contact_residues", "hotspot_contacts",
          "hotspot_fraction"):
    if c not in d.columns:
        d[c] = pd.NA
d["contact_residues"] = d["contact_residues"].astype("object")
_done = 0
for i in d.index:
    _cif = f"{OUT}/structures/{d.at[i, 'peptide_id']}.cif"
    _m = measure(_cif) if os.path.isfile(_cif) else None
    if _m:
        for k, v in _m.items():
            d.at[i, k] = v
        _done += 1
if _done:
    print(f"re-measured {_done} design(s): contacts within {contact_cutoff} A, "
          f"{len(HOT)} hotspot(s), {len(PATCH)} residue(s) in the patch")
else:
    print("no saved structures found, so step 4's contact numbers are used as they "
          "are -- changing the hotspots or either radius here will do nothing")
for c in ("n_contact_res", "hotspot_contacts", "hotspot_fraction"):
    d[c] = pd.to_numeric(d[c], errors="coerce")

# ------------------------------------------------------------ sequence properties
KD = dict(A=1.8, R=-4.5, N=-3.5, D=-3.5, C=2.5, Q=-3.5, E=-3.5, G=-0.4, H=-3.2,
          I=4.5, L=3.8, K=-3.9, M=1.9, F=2.8, P=-1.6, S=-0.8, T=-0.7, W=-0.9,
          Y=-1.3, V=4.2)
LIAB = [("N-terminal Gln", lambda s: s.startswith("Q")),
        ("Asp-Pro (acid-labile)", lambda s: "DP" in s),
        ("Asn-Gly (deamidation)", lambda s: "NG" in s),
        ("Asp-Gly (isomerisation)", lambda s: "DG" in s),
        ("Met (oxidation)", lambda s: "M" in s),
        ("4+ identical in a row", lambda s: bool(re.search(r"(.)\1{3,}", s))),
        ("3+ Pro in a row", lambda s: "PPP" in s)]


def props(s):
    pos = s.count("K") + s.count("R") + 0.1 * s.count("H")
    flags = [n for n, f in LIAB if f(s)]
    return pd.Series({
        "n_cys": s.count("C"),
        "net_charge": round(pos - s.count("D") - s.count("E"), 1),
        "gravy": round(float(np.mean([KD[c] for c in s])), 3),
        "liabilities": "; ".join(flags),
        "n_liabilities": len(flags)})


d = pd.concat([d, d["sequence"].apply(props)], axis=1)

# ------------------------------------------------------------ the gates
G = []
G.append(("iptm", d["iptm_pair"].astype(float) >= float(min_iptm)))
if "ipae" in d.columns:
    G.append(("ipae", d["ipae"].astype(float) <= float(max_ipae)))
if "pep_plddt" in d.columns:
    G.append(("plddt", d["pep_plddt"].astype(float) >= float(min_pep_plddt)))
if float(max_receptor_rmsd) > 0 and "receptor_rmsd" in d.columns:
    r = pd.to_numeric(d["receptor_rmsd"], errors="coerce")
    G.append(("receptor_moved", r.isna() | (r <= float(max_receptor_rmsd))))
if HOT:
    G.append(("hotspot_contacts",
              d["hotspot_contacts"].astype(float) >= int(min_hotspot_contacts)))
    G.append(("wrong_face",
              d["hotspot_fraction"].astype(float) >= float(min_hotspot_fraction)))
else:
    print("no hotspots were given in step 2, so the two hotspot gates are skipped")
G.append(("cys", d["n_cys"] <= int(max_cys)))
G.append(("charge", d["net_charge"].between(float(charge_min), float(charge_max))))
G.append(("gravy", d["gravy"] <= float(max_gravy)))
if drop_liabilities:
    G.append(("liabilities", d["n_liabilities"] == 0))

failed = []
for i in d.index:
    failed.append(", ".join(n for n, m in G if not bool(m.loc[i])))
d["failed"] = failed
d["pass_all"] = d["failed"] == ""

print(f"{len(d)} cofolded designs\n")
print(f"{'gate':<20s} {'passing':>8s}")
print("-" * 30)
for n, m in G:
    print(f"{n:<20s} {int(m.sum()):>5d}/{len(d)}")
print("-" * 30)
print(f"{'ALL GATES':<20s} {int(d['pass_all'].sum()):>5d}/{len(d)}")

asc = rank_by in ("ipae", "perplexity")
final = d[d["pass_all"]].sort_values(rank_by, ascending=asc).reset_index(drop=True)
final.insert(0, "rank", np.arange(1, len(final) + 1))

show = [c for c in ["rank", "peptide_id", "length", "sequence", "perplexity",
                    "iptm_pair", "ipae", "pep_plddt", "hotspot_contacts",
                    "hotspot_fraction", "net_charge", "gravy", "n_cys",
                    "liabilities"] if c in final.columns]
if len(final):
    print(f"\n{len(final)} passing, ranked by {rank_by}:\n")
    print(final[show].head(25).to_string(index=False))
    shutil.rmtree(f"{OUT}/passing", ignore_errors=True)
    os.makedirs(f"{OUT}/passing", exist_ok=True)
    for _, r in final.iterrows():
        s = f"{OUT}/structures/{r['peptide_id']}.cif"
        if os.path.isfile(s):
            shutil.copyfile(s, f"{OUT}/passing/rank{int(r['rank']):03d}_"
                               f"{r['peptide_id']}.cif")
else:
    print("\nNothing passed. The gate table above shows where they died.")
    near = d.nlargest(min(10, len(d)), "iptm_pair")
    print("\nthe 10 most confident, and what each one failed:\n")
    print(near[[c for c in show if c in near.columns] + ["failed"]]
          .to_string(index=False))

final.to_csv(f"{OUT}/final.csv", index=False)
final.to_csv(f"{WORK}/final.csv", index=False)

# master.csv must keep EVERY generated peptide, so merge this cell's columns back into
# the full table. Writing `d` here would silently drop every peptide that was never
# selected for cofolding.
_full = df.set_index("peptide_id")
_sub = d.set_index("peptide_id")
for c in _sub.columns:
    if c not in _full.columns:
        _full[c] = pd.NA
_full.update(_sub)
_full = _full.reset_index()
assert len(_full) == len(df), f"master.csv would lose rows ({len(_full)} vs {len(df)})"
_full.to_csv(f"{OUT}/master.csv", index=False)
_full.to_csv(f"{WORK}/master.csv", index=False)
json.dump({"min_iptm": float(min_iptm), "max_ipae": float(max_ipae),
           "min_pep_plddt": float(min_pep_plddt),
           "max_receptor_rmsd": float(max_receptor_rmsd),
           "min_hotspot_contacts": int(min_hotspot_contacts),
           "min_hotspot_fraction": float(min_hotspot_fraction),
           "max_cys": int(max_cys), "charge": [float(charge_min), float(charge_max)],
           "max_gravy": float(max_gravy), "drop_liabilities": bool(drop_liabilities),
           "rank_by": rank_by, "n_pass": int(len(final)),
           "hotspots_raw": hotspots.strip(), "hotspots_idx": HOT,
           "patch_radius": float(patch_radius),
           "contact_cutoff": float(contact_cutoff)},
          open(f"{OUT}/filter_settings.json", "w"), indent=2)
print(f"\nmaster.csv  {len(_full)} peptides generated, {len(d)} of them cofolded, "
      f"every metric, unfiltered")
print(f"final.csv   {len(final)} passing, ranked")
print(f"both in My Drive/{E['drive_folder']}/{T['run']}/")

In [ ]:
#@title 6 &middot; Look at a result { display-mode: "form" }
which = "passing (final.csv)"  #@param ["passing (final.csv)", "any cofolded (master.csv)"]
#@markdown Which row to show: rank 1 is the best by the ranking you chose in step 5.
rank = 1  #@param {type:"integer"}
colour_by = "pLDDT"  #@param ["pLDDT", "plain"]
show_contacts = True  #@param {type:"boolean"}

import json, os
import pandas as pd
import py3Dmol

E = json.load(open("/content/pepmlm/env.json"))
T = json.load(open("/content/pepmlm/target.json"))
WORK, OUT = E["work"], T["out"]
# whichever hotspots step 5 last judged against
_fs = f"{OUT}/filter_settings.json"
HOT = json.load(open(_fs)).get("hotspots_idx", []) if os.path.isfile(_fs) else []

if which.startswith("passing"):
    d = pd.read_csv(f"{WORK}/final.csv")
    assert len(d), "final.csv is empty -- loosen the gates in step 5"
else:
    d = pd.read_csv(f"{WORK}/master.csv")
    d = d[d["cofolded"].eq(True)].copy()
    d["iptm_pair"] = pd.to_numeric(d["iptm_pair"], errors="coerce")
    d = d.sort_values("iptm_pair", ascending=False).reset_index(drop=True)
    d.insert(0, "rank", range(1, len(d) + 1))

i = max(1, min(int(rank), len(d))) - 1
r = d.iloc[i]
cif = f"{OUT}/structures/{r['peptide_id']}.cif"
assert os.path.isfile(cif), f"no structure saved for {r['peptide_id']}"

print(f"rank {int(r.get('rank', i + 1))} of {len(d)}   {r['peptide_id']}")
print(f"  sequence        {r['sequence']}   ({int(r['length'])} aa)")
print(f"  perplexity      {r['perplexity']:.3f}")
print(f"  interface ipTM  {r['iptm_pair']:.3f}"
      + (f"   ipAE {r['ipae']:.2f} A" if "ipae" in r and pd.notna(r["ipae"]) else ""))
if "pep_plddt" in r and pd.notna(r["pep_plddt"]):
    print(f"  peptide pLDDT   {float(r['pep_plddt']):.3f}")
if HOT:
    print(f"  hotspots hit    {int(r['hotspot_contacts'])} of {len(HOT)}"
          f"   on-patch fraction {float(r['hotspot_fraction']):.2f}")
if "contact_residues" in r and isinstance(r["contact_residues"], str):
    lab = T["labels"]
    got = [int(x) for x in r["contact_residues"].split(";") if x]
    print(f"  touches         "
          + ", ".join(f"{T['seq'][j - 1]}{lab[j - 1]}" for j in got[:18])
          + (" ..." if len(got) > 18 else ""))
    print("                  (your own numbering)")
if "liabilities" in r and isinstance(r["liabilities"], str) and r["liabilities"]:
    print(f"  watch out for   {r['liabilities']}")

v = py3Dmol.view(width=820, height=520)
v.addModel(open(cif).read(), "cif")
v.setStyle({"chain": "A"}, {"cartoon": {"color": "grey", "opacity": 0.8}})
if colour_by == "pLDDT":
    v.setStyle({"chain": "B"}, {"cartoon": {"colorscheme": {"prop": "b",
                "gradient": "roygb", "min": 0.5, "max": 1.0}}})
    v.addStyle({"chain": "B"}, {"stick": {"radius": 0.16}})
else:
    v.setStyle({"chain": "B"}, {"cartoon": {"color": "orange"},
                                "stick": {"radius": 0.16}})
if HOT:
    v.addStyle({"chain": "A", "resi": [str(x) for x in HOT]},
               {"stick": {"colorscheme": "pinkCarbon", "radius": 0.28}})
if show_contacts and "contact_residues" in r and isinstance(r["contact_residues"], str):
    cr = [x for x in r["contact_residues"].split(";") if x]
    v.addStyle({"chain": "A", "resi": cr},
               {"cartoon": {"color": "lightblue"}})
v.zoomTo({"chain": "B"})
v.show()
msg = "grey = target, "
msg += "peptide red->blue by pLDDT" if colour_by == "pLDDT" else "orange = peptide"
print("\n" + msg + ", pink sticks = your hotspots"
      + (", pale blue = residues the peptide touches" if show_contacts else ""))

## Reading the numbers

| Column | What it is | Rough reading |
|---|---|---|
| `perplexity` | PepMLM's own confidence. **Not comparable between lengths.** | lower is better |
| `iptm_pair` | Boltz-2 interface confidence, receptor&ndash;peptide specifically | >0.6 interesting, >0.75 good |
| `ipae` | interface predicted error, &aring;ngstroms | <10 interesting, <7 good |
| `pep_plddt` | is the peptide itself confidently placed | >0.7 |
| `hotspot_contacts` | how many of step 5's residues it actually touches | the more the better |
| `hotspot_fraction` | share of its contacts on the patch around your hotspots | <0.5 means it is on the wrong face |
| `receptor_rmsd` | how far the receptor moved vs. the free prediction | large = a pose that needs the target to deform |
| `net_charge`, `gravy`, `n_cys`, `liabilities` | can you actually make it | see step 5 |

## What this is not

A peptide that passes every gate here has passed a **structure-prediction** check, not
a binding assay. Boltz-2 was trained on the PDB, which overlaps the data PepMLM was
trained on, so some of its confidence is circularity rather than physics. On short
linear peptides its ipTM is a much weaker signal than it is on folded miniproteins.

Treat `final.csv` as a shortlist worth synthesising, in which most entries will still
not bind. If you need higher prior odds per peptide, a structure-based designer
(RFdiffusion, BindCraft, PXDesign) that builds a folded miniprotein is the better
starting point — at the cost of a molecule you cannot make by solid-phase synthesis.

## Re-folding the finalists properly

Step 4 runs at `sampling_steps = 50` to screen cheaply. Once `final.csv` is short,
raise it to Boltz's default **200**, set `diffusion_samples` to 3&ndash;5, tick
`redo`, and run step 4 again on just the finalists (set `max_designs` to their count
after re-running step 3 with a tighter gate). Metrics that survive that are worth
more than metrics that only existed at 50 steps.

## Credit

- **PepMLM** &mdash; Chen, Quinn, Dumas et al., *Target sequence-conditioned design of
  peptide binders using masked language modeling*, Nature Biotechnology (2025).
  [repo](https://github.com/programmablebio/pepmlm) &middot;
  [paper](https://www.nature.com/articles/s41587-025-02761-2)
- **Boltz-2** &mdash; Passaro, Corso, Wohlwend et al. (2025).
  [repo](https://github.com/jwohlwend/boltz)
- MSAs come from the ColabFold MMseqs2 server. Please do not hammer it: this notebook
  queries it **once per target sequence** and caches the result in your Drive.